# 95 — Estimate nodal source positions, then match to Geode metadata

This notebook separates two tasks:

1. **Estimate source position for each nodal event** using the nodal shot gather only.
2. **Compare nodal event times/positions with Geode metadata** using a fixed, user-controlled clock offset per survey.

This avoids hidden offset-hypothesis searches. The goal is to make the timing/position relationship visible first, then apply deterministic matching once the Geode clock offset is understood.

In [ ]:
from pathlib import Path
import json
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read

try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception as e:
    PLOTLY_AVAILABLE = False
    print("Plotly not available; falling back to matplotlib where possible:", e)

## 1. Configuration

In [ ]:
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"

OUTPUT_DIR = PROJECT_ROOT / "catalog" / "nodal_geode_matching"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NODAL_TIMEWINDOW_LABEL = "T1_N2_Refraction1m"
TARGET_GEODE_SURVEY = "T1_1m_refraction"

# Sign convention:
# corrected_geode_time = geode_recorded_time + GEODE_FIXED_OFFSET_S
#
# If Geode/laptop clock was 11.5 s ahead of true UTC, the recorded time is too late,
# so corrected UTC = recorded time - 11.5 s.
GEODE_FIXED_OFFSET_S = -11.5

DEFAULT_STACK_BEFORE_S = 5.0
DEFAULT_STACK_AFTER_S = 120.0
SECONDS_PER_BLOW_GUESS = 6.0
MIN_STACK_AFTER_S = 30.0
MAX_STACK_AFTER_S = 180.0

SOURCE_POSITION_COMPONENT = "Z"
SOURCE_HP_FREQ_HZ = 25.0
SOURCE_HP_CORNERS = 4
SOURCE_ENERGY_TMIN_S = 0.0
SOURCE_ENERGY_TMAX_S = None
SOURCE_TOP_N = 3
SOURCE_RELATIVE_THRESHOLD = 0.10

MAX_EVENTS_TO_ESTIMATE = None
MAX_MATCHES_TO_DISPLAY = 50

WRITE_SOURCE_ESTIMATES_TO_DB = True
WRITE_MATCH_CANDIDATES_TO_DB = True

print("Catalog:", CATALOG_DB)
print("Output:", OUTPUT_DIR)

## 2. Connect to SQLite catalog and load tables

In [ ]:
if not CATALOG_DB.exists():
    raise FileNotFoundError(CATALOG_DB)

conn = sqlite3.connect(CATALOG_DB)

def table_exists(name):
    q = "SELECT name FROM sqlite_master WHERE type='table' AND name=?"
    return conn.execute(q, (name,)).fetchone() is not None

for table in ["shot_events", "shot_gather_files", "trace_index", "picks"]:
    print(table, "exists =", table_exists(table))

In [ ]:
shot_events = pd.read_sql("SELECT * FROM shot_events", conn)
shot_gather_files = pd.read_sql("SELECT * FROM shot_gather_files", conn)
trace_index = pd.read_sql("SELECT * FROM trace_index", conn)
picks = pd.read_sql("SELECT * FROM picks", conn) if table_exists("picks") else pd.DataFrame()

print("shot_events:", len(shot_events))
print("shot_gather_files:", len(shot_gather_files))
print("trace_index:", len(trace_index))
print("picks:", len(picks))

display(shot_events.head())

In [ ]:
def parse_datetime_col(df, candidates, out_col):
    df = df.copy()
    for c in candidates:
        if c in df.columns:
            dt = pd.to_datetime(df[c], errors="coerce", utc=True)
            if dt.notna().any():
                df[out_col] = dt
                return df
    df[out_col] = pd.NaT
    return df

shot_events = parse_datetime_col(
    shot_events,
    ["event_time_utc", "shot_time_utc", "detection_time_utc", "time_utc"],
    "event_time_dt",
)

shot_events = parse_datetime_col(
    shot_events,
    ["shot_time_utc", "event_time_utc", "detection_time_utc", "file_time_utc"],
    "shot_time_dt",
)

cols = [c for c in ["event_id", "instrument_system", "survey", "line", "timewindow_label", "event_time_dt", "shot_time_dt", "source_x_m"] if c in shot_events.columns]
display(shot_events[cols].head(10))

## 3. Select nodal and Geode subsets

In [ ]:
nodal_events = shot_events[
    shot_events.get("instrument_system", "").astype(str).str.lower().eq("nodal")
].copy()

if "timewindow_label" not in nodal_events.columns:
    raise ValueError("shot_events has no timewindow_label column")

nodal_subset = nodal_events[
    nodal_events["timewindow_label"].astype(str).eq(TARGET_NODAL_TIMEWINDOW_LABEL)
].copy()

nodal_subset = nodal_subset.sort_values("event_time_dt").reset_index(drop=True)

if MAX_EVENTS_TO_ESTIMATE is not None:
    nodal_subset = nodal_subset.head(MAX_EVENTS_TO_ESTIMATE)

print("Target nodal time window:", TARGET_NODAL_TIMEWINDOW_LABEL)
print("Nodal events selected:", len(nodal_subset))
display(nodal_subset[["event_id", "line", "timewindow_label", "event_time_dt", "source_x_m"]].head(20))

In [ ]:
geode_events = shot_events[
    ~shot_events.get("instrument_system", "").astype(str).str.lower().eq("nodal")
].copy()

if TARGET_GEODE_SURVEY is not None:
    geode_subset = geode_events[
        geode_events.get("survey", "").astype(str).str.lower().eq(TARGET_GEODE_SURVEY.lower())
    ].copy()
else:
    geode_subset = geode_events.copy()

geode_subset = geode_subset.sort_values("event_time_dt").reset_index(drop=True)

print("Target Geode survey:", TARGET_GEODE_SURVEY)
print("Geode events selected:", len(geode_subset))
cols = [c for c in ["event_id", "survey", "line", "event_time_dt", "shot_time_dt", "source_x_m", "file_no", "n_blows"] if c in geode_subset.columns]
display(geode_subset[cols].head(30))

print("Geode rows with usable event_time_dt:", geode_subset["event_time_dt"].notna().sum())
print("Geode rows with usable source_x_m:", pd.to_numeric(geode_subset.get("source_x_m", np.nan), errors="coerce").notna().sum())

## 4. Helper functions: load nodal gather and estimate source x

In [ ]:
def load_nodal_mseed_for_event(event_id):
    q = '''
    SELECT file_path
    FROM shot_gather_files
    WHERE event_id = ?
      AND instrument_system = 'nodal'
      AND file_type = 'mseed'
    ORDER BY file_path
    '''
    df = pd.read_sql(q, conn, params=[str(event_id)])

    if df.empty:
        raise ValueError(f"No nodal MiniSEED file found for {event_id}")

    path = Path(df.iloc[0]["file_path"])

    if not path.exists():
        raise FileNotFoundError(path)

    return read(str(path)), path


def trace_receiver_x(event_id, tr):
    if not trace_index.empty:
        mask = (
            trace_index["event_id"].astype(str).eq(str(event_id))
            & trace_index["station"].astype(str).eq(str(tr.stats.station))
        )

        if "channel" in trace_index.columns:
            mask = mask & trace_index["channel"].astype(str).eq(str(tr.stats.channel))

        rows = trace_index[mask]

        if len(rows) and "receiver_x_m" in rows.columns:
            return pd.to_numeric(rows.iloc[0]["receiver_x_m"], errors="coerce")

    try:
        return float(tr.stats.station) / 100.0
    except Exception:
        return np.nan

In [ ]:
def estimate_source_x_from_gather(
    event_id,
    st,
    component=SOURCE_POSITION_COMPONENT,
    hp_freq_hz=SOURCE_HP_FREQ_HZ,
    hp_corners=SOURCE_HP_CORNERS,
    energy_tmin_s=SOURCE_ENERGY_TMIN_S,
    energy_tmax_s=SOURCE_ENERGY_TMAX_S,
    top_n=SOURCE_TOP_N,
    relative_threshold=SOURCE_RELATIVE_THRESHOLD,
):
    """
    Estimate source x from a nodal shot gather using high-pass filtered energy.

    Method:
      1. Select one component, usually Z.
      2. High-pass filter each trace to suppress low-frequency ground roll.
      3. Compute energy over the event window.
      4. Estimate source x from an energy-weighted average of the strongest receivers.
    """

    rows = []
    t0 = min(tr.stats.starttime for tr in st)

    for tr in st:
        ch = tr.stats.channel

        if component and not ch.endswith(component):
            continue

        x_m = trace_receiver_x(event_id, tr)

        if not np.isfinite(x_m) or tr.stats.npts == 0:
            continue

        dt = float(tr.stats.delta)

        tr_work = tr.copy()
        tr_work.detrend("linear")
        tr_work.taper(max_percentage=0.02)

        rel_start_s = tr_work.stats.starttime - t0

        i1 = max(0, int(round((energy_tmin_s - rel_start_s) / dt)))

        if energy_tmax_s is None:
            i2 = tr_work.stats.npts
        else:
            i2 = min(tr_work.stats.npts, int(round((energy_tmax_s - rel_start_s) / dt)))

        if i2 <= i1 + 5:
            continue

        raw = tr_work.data[i1:i2].astype(float)
        raw = raw - np.nanmedian(raw)

        tr_hp = tr_work.copy()
        tr_hp.filter("highpass", freq=hp_freq_hz, corners=hp_corners, zerophase=True)
        hp = tr_hp.data[i1:i2].astype(float)
        hp = hp - np.nanmedian(hp)

        rows.append({
            "station": tr.stats.station,
            "channel": ch,
            "receiver_x_m": float(x_m),
            "energy_raw": float(np.nansum(raw**2)),
            "energy_hp25": float(np.nansum(hp**2)),
            "rms_raw": float(np.sqrt(np.nanmean(raw**2))),
            "rms_hp25": float(np.sqrt(np.nanmean(hp**2))),
            "peak_abs_raw": float(np.nanmax(np.abs(raw))) if len(raw) else np.nan,
            "peak_abs_hp25": float(np.nanmax(np.abs(hp))) if len(hp) else np.nan,
        })

    metric = pd.DataFrame(rows)

    if metric.empty:
        return {
            "estimated_source_x_m": np.nan,
            "estimation_method": "none",
            "energy_x_m": np.nan,
            "metric": metric,
            "top_energy_receivers": [],
        }

    metric["receiver_x_m"] = pd.to_numeric(metric["receiver_x_m"], errors="coerce")
    metric["energy_hp25"] = pd.to_numeric(metric["energy_hp25"], errors="coerce")
    metric = metric.dropna(subset=["receiver_x_m", "energy_hp25"])

    if metric.empty:
        return {
            "estimated_source_x_m": np.nan,
            "estimation_method": "none_after_cleaning",
            "energy_x_m": np.nan,
            "metric": metric,
            "top_energy_receivers": [],
        }

    metric_sorted = metric.sort_values("energy_hp25", ascending=False).copy()
    emax = metric_sorted["energy_hp25"].max()

    top = metric_sorted[metric_sorted["energy_hp25"] >= relative_threshold * emax].copy()

    if len(top) < 2:
        top = metric_sorted.head(top_n).copy()

    xs = top["receiver_x_m"].to_numpy(dtype=float)
    weights = top["energy_hp25"].to_numpy(dtype=float)

    if np.nansum(weights) > 0:
        estimated_x = float(np.nansum(xs * weights) / np.nansum(weights))
    else:
        estimated_x = float(metric_sorted.iloc[0]["receiver_x_m"])

    energy_x = float(metric_sorted.iloc[0]["receiver_x_m"])

    top_records = top[
        ["station", "channel", "receiver_x_m", "energy_hp25", "rms_hp25", "peak_abs_hp25"]
    ].to_dict("records")

    return {
        "estimated_source_x_m": estimated_x,
        "estimation_method": f"weighted_energy_hp{int(hp_freq_hz)}",
        "energy_x_m": energy_x,
        "metric": metric,
        "top_energy_receivers": top_records,
    }

## 5. Estimate source position for nodal events

In [ ]:
source_estimate_rows = []

for i, row in nodal_subset.iterrows():
    event_id = row["event_id"]
    print(f"[{i+1}/{len(nodal_subset)}] {event_id}")

    try:
        st, mseed_path = load_nodal_mseed_for_event(event_id)

        est = estimate_source_x_from_gather(
            event_id,
            st,
            component=SOURCE_POSITION_COMPONENT,
        )

        source_estimate_rows.append({
            "event_id": event_id,
            "line": row.get("line"),
            "timewindow_label": row.get("timewindow_label"),
            "event_time_utc": row.get("event_time_dt").isoformat() if pd.notna(row.get("event_time_dt")) else None,
            "estimated_source_x_m": est["estimated_source_x_m"],
            "estimation_method": est["estimation_method"],
            "energy_x_m": est["energy_x_m"],
            "top_energy_receivers_json": json.dumps(est["top_energy_receivers"]),
            "mseed_path": str(mseed_path),
            "error": None,
        })

    except Exception as e:
        print("  FAILED:", e)
        source_estimate_rows.append({
            "event_id": event_id,
            "line": row.get("line"),
            "timewindow_label": row.get("timewindow_label"),
            "event_time_utc": row.get("event_time_dt").isoformat() if pd.notna(row.get("event_time_dt")) else None,
            "estimated_source_x_m": np.nan,
            "estimation_method": "failed",
            "energy_x_m": np.nan,
            "top_energy_receivers_json": "[]",
            "mseed_path": None,
            "error": str(e),
        })

source_estimates = pd.DataFrame(source_estimate_rows)
source_estimates["event_time_dt"] = pd.to_datetime(source_estimates["event_time_utc"], errors="coerce", utc=True)

print("Source estimates:", len(source_estimates))
display(source_estimates.head())
display(source_estimates["estimated_source_x_m"].describe())

In [ ]:
source_est_csv = OUTPUT_DIR / f"nodal_source_estimates_{TARGET_NODAL_TIMEWINDOW_LABEL}.csv"
source_estimates.to_csv(source_est_csv, index=False)
print("Wrote:", source_est_csv)

if WRITE_SOURCE_ESTIMATES_TO_DB:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS nodal_source_estimates (
            event_id TEXT PRIMARY KEY,
            line TEXT,
            timewindow_label TEXT,
            event_time_utc TEXT,
            estimated_source_x_m REAL,
            estimation_method TEXT,
            energy_x_m REAL,
            top_energy_receivers_json TEXT,
            mseed_path TEXT,
            error TEXT
        )
    ''')

    conn.execute(
        "DELETE FROM nodal_source_estimates WHERE timewindow_label = ?",
        (TARGET_NODAL_TIMEWINDOW_LABEL,),
    )

    db_cols = [
        "event_id",
        "line",
        "timewindow_label",
        "event_time_utc",
        "estimated_source_x_m",
        "estimation_method",
        "energy_x_m",
        "top_energy_receivers_json",
        "mseed_path",
        "error",
    ]

    source_estimates[db_cols].to_sql(
        "nodal_source_estimates",
        conn,
        if_exists="append",
        index=False,
    )
    conn.commit()
    print("Updated SQLite table: nodal_source_estimates")

## 6. Visualize nodal source position versus time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.scatter(
    source_estimates["event_time_dt"],
    source_estimates["estimated_source_x_m"],
    s=20,
    label="Nodal estimated source x",
)

ax.set_title(f"Nodal source-position estimates: {TARGET_NODAL_TIMEWINDOW_LABEL}")
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Estimated source x (m)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

In [ ]:
if PLOTLY_AVAILABLE:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=source_estimates["event_time_dt"],
        y=source_estimates["estimated_source_x_m"],
        mode="markers",
        name="Nodal source estimates",
        marker=dict(size=6),
        text=source_estimates["event_id"],
        hovertemplate=(
            "event=%{text}<br>"
            "time=%{x}<br>"
            "estimated x=%{y:.2f} m<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=f"Zoomable nodal source-position estimates: {TARGET_NODAL_TIMEWINDOW_LABEL}",
        xaxis_title="Time (UTC)",
        yaxis_title="Estimated source x (m)",
        hovermode="closest",
        height=600,
    )

    fig.show()
else:
    print("Plotly not available.")

## 7. Compare nodal source estimates to Geode file times/positions

In [ ]:
geode_plot = geode_subset.copy()
geode_plot["source_x_m"] = pd.to_numeric(geode_plot.get("source_x_m"), errors="coerce")
geode_plot["event_time_corrected_dt"] = geode_plot["event_time_dt"] + pd.to_timedelta(GEODE_FIXED_OFFSET_S, unit="s")

cols = [c for c in ["event_id", "survey", "file_no", "event_time_dt", "event_time_corrected_dt", "source_x_m", "n_blows"] if c in geode_plot.columns]
display(geode_plot[cols].head(30))

print("Fixed Geode offset applied:", GEODE_FIXED_OFFSET_S, "s")

In [ ]:
if PLOTLY_AVAILABLE:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=source_estimates["event_time_dt"],
        y=source_estimates["estimated_source_x_m"],
        mode="markers",
        name="Nodal detections",
        marker=dict(size=6, color="blue"),
        text=source_estimates["event_id"],
        hovertemplate=(
            "nodal event=%{text}<br>"
            "time=%{x}<br>"
            "estimated x=%{y:.2f} m<extra></extra>"
        ),
    ))

    fig.add_trace(go.Scatter(
        x=geode_plot["event_time_dt"],
        y=geode_plot["source_x_m"],
        mode="markers",
        name="Geode recorded times",
        marker=dict(size=9, color="red", symbol="x"),
        text=geode_plot["event_id"],
        hovertemplate=(
            "geode event=%{text}<br>"
            "recorded time=%{x}<br>"
            "source x=%{y:.2f} m<extra></extra>"
        ),
    ))

    fig.add_trace(go.Scatter(
        x=geode_plot["event_time_corrected_dt"],
        y=geode_plot["source_x_m"],
        mode="markers",
        name=f"Geode corrected times ({GEODE_FIXED_OFFSET_S:+.1f} s)",
        marker=dict(size=9, color="orange", symbol="diamond"),
        text=geode_plot["event_id"],
        hovertemplate=(
            "geode event=%{text}<br>"
            "corrected time=%{x}<br>"
            "source x=%{y:.2f} m<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=f"Nodal detections vs Geode metadata: {TARGET_NODAL_TIMEWINDOW_LABEL} / {TARGET_GEODE_SURVEY}",
        xaxis_title="Time",
        yaxis_title="Source position x (m)",
        hovermode="closest",
        height=700,
    )

    fig.show()
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.scatter(source_estimates["event_time_dt"], source_estimates["estimated_source_x_m"], s=20, label="Nodal")
    ax.scatter(geode_plot["event_time_dt"], geode_plot["source_x_m"], s=40, marker="x", label="Geode recorded")
    ax.scatter(geode_plot["event_time_corrected_dt"], geode_plot["source_x_m"], s=40, marker="D", label="Geode corrected")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

## 8. Estimate fixed Geode clock offset from first nodal event near each shot position

In [ ]:
def nearest_nodal_events_for_geode(
    geode_row,
    source_estimates,
    max_dx_m=3.0,
    max_dt_s=300.0,
):
    gx = pd.to_numeric(pd.Series([geode_row.get("source_x_m", np.nan)]), errors="coerce").iloc[0]
    gt = geode_row.get("event_time_dt", pd.NaT)

    if not np.isfinite(gx) or pd.isna(gt):
        return pd.DataFrame()

    df = source_estimates.copy()
    df["estimated_source_x_m"] = pd.to_numeric(df["estimated_source_x_m"], errors="coerce")
    df = df.dropna(subset=["estimated_source_x_m", "event_time_dt"])

    df["source_x_residual_m"] = df["estimated_source_x_m"] - gx
    df["time_residual_recorded_s"] = (df["event_time_dt"] - gt).dt.total_seconds()

    df = df[
        (df["source_x_residual_m"].abs() <= max_dx_m)
        & (df["time_residual_recorded_s"].abs() <= max_dt_s)
    ].copy()

    return df.sort_values(["time_residual_recorded_s", "source_x_residual_m"])


clock_rows = []

for _, gr in geode_plot.iterrows():
    near = nearest_nodal_events_for_geode(
        gr,
        source_estimates,
        max_dx_m=3.0,
        max_dt_s=300.0,
    )

    if near.empty:
        continue

    first = near.sort_values("event_time_dt").iloc[0]

    clock_rows.append({
        "geode_event_id": gr.get("event_id"),
        "geode_file_no": gr.get("file_no"),
        "geode_source_x_m": gr.get("source_x_m"),
        "geode_recorded_time_utc": gr.get("event_time_dt").isoformat() if pd.notna(gr.get("event_time_dt")) else None,
        "first_nodal_event_id": first["event_id"],
        "first_nodal_time_utc": first["event_time_dt"].isoformat(),
        "first_nodal_estimated_x_m": first["estimated_source_x_m"],
        "source_x_residual_m": first["source_x_residual_m"],
        "nodal_minus_geode_recorded_s": first["time_residual_recorded_s"],
        "implied_geode_offset_s": first["time_residual_recorded_s"],
        "n_nodal_events_near_source": len(near),
    })

clock_offsets = pd.DataFrame(clock_rows)

print("Clock-offset estimates:", len(clock_offsets))
display(clock_offsets.head(30))

if len(clock_offsets):
    display(clock_offsets["implied_geode_offset_s"].describe())

In [ ]:
if len(clock_offsets):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(
        clock_offsets["geode_source_x_m"],
        clock_offsets["implied_geode_offset_s"],
        "o-",
    )
    ax.axhline(GEODE_FIXED_OFFSET_S, color="red", linestyle="--", label=f"current offset {GEODE_FIXED_OFFSET_S:+.1f} s")
    ax.set_xlabel("Geode source x (m)")
    ax.set_ylabel("Implied offset: nodal time - Geode recorded time (s)")
    ax.set_title("Estimated Geode clock offset from first nodal event near each shot position")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

    clock_csv = OUTPUT_DIR / f"clock_offset_estimates_{TARGET_GEODE_SURVEY}.csv"
    clock_offsets.to_csv(clock_csv, index=False)
    print("Wrote:", clock_csv)

## 9. Deterministic matching using fixed offset and stack window

In [ ]:
def stack_after_s(row):
    n_blows = row.get("n_blows", np.nan)
    try:
        n_blows = float(n_blows)
    except Exception:
        n_blows = np.nan

    if np.isfinite(n_blows) and n_blows > 1:
        return min(
            MAX_STACK_AFTER_S,
            max(MIN_STACK_AFTER_S, n_blows * SECONDS_PER_BLOW_GUESS),
        )

    return DEFAULT_STACK_AFTER_S


def match_nodal_to_geode_fixed_offset(
    source_estimates,
    geode_plot,
    source_tolerance_m=3.0,
):
    rows = []

    nodal = source_estimates.copy()
    nodal["estimated_source_x_m"] = pd.to_numeric(nodal["estimated_source_x_m"], errors="coerce")
    nodal = nodal.dropna(subset=["estimated_source_x_m", "event_time_dt"])

    for _, gr in geode_plot.iterrows():
        gt = gr.get("event_time_corrected_dt", pd.NaT)
        gx = pd.to_numeric(pd.Series([gr.get("source_x_m", np.nan)]), errors="coerce").iloc[0]

        if pd.isna(gt) or not np.isfinite(gx):
            continue

        before_s = DEFAULT_STACK_BEFORE_S
        after_s = stack_after_s(gr)

        t1 = gt - pd.to_timedelta(before_s, unit="s")
        t2 = gt + pd.to_timedelta(after_s, unit="s")

        cand = nodal[
            (nodal["event_time_dt"] >= t1)
            & (nodal["event_time_dt"] <= t2)
        ].copy()

        if cand.empty:
            continue

        cand["source_x_residual_m"] = cand["estimated_source_x_m"] - gx
        cand["time_residual_s"] = (cand["event_time_dt"] - gt).dt.total_seconds()

        cand = cand[cand["source_x_residual_m"].abs() <= source_tolerance_m].copy()

        if cand.empty:
            continue

        for _, nr in cand.sort_values("event_time_dt").iterrows():
            rows.append({
                "nodal_event_id": nr["event_id"],
                "nodal_time_utc": nr["event_time_dt"].isoformat(),
                "nodal_estimated_source_x_m": nr["estimated_source_x_m"],
                "nodal_timewindow_label": nr["timewindow_label"],

                "geode_event_id": gr.get("event_id"),
                "geode_survey": gr.get("survey"),
                "geode_file_no": gr.get("file_no"),
                "geode_recorded_time_utc": gr.get("event_time_dt").isoformat() if pd.notna(gr.get("event_time_dt")) else None,
                "geode_corrected_time_utc": gt.isoformat(),
                "geode_fixed_offset_s": GEODE_FIXED_OFFSET_S,
                "geode_source_x_m": gx,

                "source_type": gr.get("source_type"),
                "operator": gr.get("operator"),
                "plate_type": gr.get("plate_type"),
                "n_blows": gr.get("n_blows"),

                "time_residual_s": nr["time_residual_s"],
                "source_x_residual_m": nr["source_x_residual_m"],
                "match_status": "candidate_fixed_offset",
            })

    return pd.DataFrame(rows)


matches = match_nodal_to_geode_fixed_offset(
    source_estimates,
    geode_plot,
    source_tolerance_m=3.0,
)

print("Deterministic matches:", len(matches))
display(matches.head(MAX_MATCHES_TO_DISPLAY))

In [ ]:
match_csv = OUTPUT_DIR / f"nodal_geode_matches_{TARGET_NODAL_TIMEWINDOW_LABEL}_{TARGET_GEODE_SURVEY}.csv"
matches.to_csv(match_csv, index=False)
print("Wrote:", match_csv)

if WRITE_MATCH_CANDIDATES_TO_DB:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS nodal_geode_match_candidates (
            nodal_event_id TEXT,
            nodal_time_utc TEXT,
            nodal_estimated_source_x_m REAL,
            nodal_timewindow_label TEXT,
            geode_event_id TEXT,
            geode_survey TEXT,
            geode_file_no TEXT,
            geode_recorded_time_utc TEXT,
            geode_corrected_time_utc TEXT,
            geode_fixed_offset_s REAL,
            geode_source_x_m REAL,
            source_type TEXT,
            operator TEXT,
            plate_type TEXT,
            n_blows REAL,
            time_residual_s REAL,
            source_x_residual_m REAL,
            match_status TEXT
        )
    ''')

    conn.execute(
        "DELETE FROM nodal_geode_match_candidates WHERE nodal_timewindow_label = ? AND geode_survey = ?",
        (TARGET_NODAL_TIMEWINDOW_LABEL, TARGET_GEODE_SURVEY),
    )

    if len(matches):
        matches.to_sql(
            "nodal_geode_match_candidates",
            conn,
            if_exists="append",
            index=False,
        )

    conn.commit()
    print("Updated SQLite table: nodal_geode_match_candidates")

## 10. Inspect one match or off-end shot

In [ ]:
INSPECT_EVENT_ID = matches["nodal_event_id"].iloc[0] if len(matches) else source_estimates["event_id"].iloc[0]

print("Inspecting:", INSPECT_EVENT_ID)

st, _ = load_nodal_mseed_for_event(INSPECT_EVENT_ID)
est = estimate_source_x_from_gather(INSPECT_EVENT_ID, st, component=SOURCE_POSITION_COMPONENT)
metric = est["metric"].sort_values("receiver_x_m").copy()

display(metric.sort_values("energy_hp25", ascending=False).head(12))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(metric["receiver_x_m"], metric["energy_hp25"], "o-", label="HP25 energy")
ax.axvline(est["estimated_source_x_m"], color="green", label=f"estimated {est['estimated_source_x_m']:.2f} m")

if len(matches):
    m = matches[matches["nodal_event_id"].astype(str).eq(str(INSPECT_EVENT_ID))]
    if len(m):
        gx = float(m.iloc[0]["geode_source_x_m"])
        ax.axvline(gx, color="red", linestyle="--", label=f"Geode source {gx:.2f} m")

ax.set_xlabel("Receiver x (m)")
ax.set_ylabel("HP25 energy")
ax.set_title(INSPECT_EVENT_ID)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## Notes

This notebook deliberately avoids searching arbitrary offset hypotheses. Instead:

1. Estimate source positions for nodal detections.
2. Plot nodal events and Geode file times together.
3. Estimate/fix a clock offset.
4. Apply deterministic matching using the fixed offset and source-position tolerance.

If the plot shows a different fixed offset for a different Geode laptop/survey, update `GEODE_FIXED_OFFSET_S` and rerun sections 7–10.